In [3]:
import json
import os
import time
import numpy as np
from rag_utils import build_chunks, project_paths, preview
from rag_store import Embedder, VectorStore


In [4]:
paths = project_paths()
embedder = Embedder()                                  # all-MiniLM-L6-v2
# Reuse Session 2's saved index if it is there and was built with this model.
if (paths["storage"] / "vectors.npy").exists():
    store = VectorStore.load(paths["storage"])
    if store.model_name != embedder.model_name or not len(store):
        store = None
else:
    store = None

if store is None:
    print("building the index from scratch ...")
    chunks = build_chunks(paths["documents"], chunk_size=500, chunk_overlap=50)
    store = VectorStore.from_chunks(chunks, embedder, show_progress_bar=False)
    store.save(paths["storage"])

store.build_faiss()
print(f"knowledge base ready: {len(store)} chunks, {store.dimension} dims, "
      f"model {store.model_name}")
print("sources:", sorted({c['metadata']['source'] for c in store.chunks}))

c:\Users\skhal\Desktop\RAG_From_Scratch\ven\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1040.27it/s]


knowledge base ready: 44 chunks, 384 dims, model sentence-transformers/all-MiniLM-L6-v2
sources: ['ai_course.pdf', 'documentation.txt']


## 1. The retriever


retrieval(query:str, top_k:int)_. list of chunks (best first )

In [6]:
class Retriever:
    """Question text -> the chunks most likely to contain the answer."""

    def __init__(self, store, embedder, top_k=4, min_score=0.2):
        self.store = store
        self.embedder = embedder
        self.top_k = top_k
        self.min_score = min_score

    def retrieve(self, query, top_k=None, min_score=None, source=None):
        top_k = self.top_k if top_k is None else top_k
        min_score = self.min_score if min_score is None else min_score
        # The query goes through the SAME model as the documents. Two models =
        # two unrelated vector spaces = confident nonsense.
        query_vector = self.embedder.encode_query(query)
        fetch = top_k * 5 if source else top_k          # over-fetch, then filter
        hits = self.store.search(query_vector, top_k=fetch, min_score=min_score)
        if source:
            hits = [h for h in hits if h["metadata"].get("source") == source]
        hits = hits[:top_k]
        for rank, hit in enumerate(hits, start=1):
            hit["rank"] = rank
        return hits


retriever = Retriever(store, embedder, top_k=4, min_score=0.2)

def retrieve_documents(query, top_k=4):
    """Function-style wrapper, matching the diagram."""
    return retriever.retrieve(query, top_k=top_k)


hits = retrieve_documents("What is the attention mechanism?", top_k=4)

print(f"{len(hits)} chunks retrieved\n")
for hit in hits:
    meta = hit["metadata"]
    print(f"[{hit['rank']}] score={hit['score']:.3f}  {meta['source']} "
          f"page {meta.get('page')}")
    print(f"    {preview(hit['text'], 150)}\n")

4 chunks retrieved

[1] score=0.794  ai_course.pdf page 3
    Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is proj ...

[2] score=0.496  ai_course.pdf page 2
    r; it reads the entire sequence at once and lets every token look at every other token. This makes training highly parallel on GPUs and removes the lo ...

[3] score=0.485  ai_course.pdf page 3
    en used as weights to average the value vectors. This is called scaled dot product attention. Self attention means queries, keys and values all come f ...

[4] score=0.477  ai_course.pdf page 3
    ead can track syntax while another tracks long range topic information, and the results are concatenated and projected back. In a decoder, attention i ...



## 2. Prompt construction


In [8]:
SYSTEM_PROMPT = """You are a careful assistant for a technical knowledge base.

Rules:
1. Answer using ONLY the numbered context passages provided by the user.
2. If the context does not contain the answer, reply exactly:
   "I don't know based on the provided documents."
3. Cite the passage number in square brackets after each claim, like [2].
4. Never invent facts, numbers, file names or citations.
5. Be concise: two to five sentences unless the question asks for more."""

USER_TEMPLATE = """Context passages:
{context}

Question: {question}

Answer (cite passages as [1], [2], ...):"""


def build_context(hits, max_chars=4000):
    """Retrieved chunks -> a numbered, source-tagged context block.

    The character budget is not optional. The context window is finite and every
    token costs money and latency. When the budget runs out we stop adding whole
    passages rather than truncating one mid-sentence, so everything the model
    sees is complete.
    """
    blocks, used = [], 0
    for rank, hit in enumerate(hits, start=1):
        meta = hit.get("metadata", {})
        page = f", page {meta['page']}" if meta.get("page") else ""
        header = f"[{rank}] (source: {meta.get('source', 'unknown')}{page})"
        block = f"{header}\n{hit['text'].strip()}"
        if used + len(block) > max_chars and blocks:      # keep at least one passage
            break
        blocks.append(block)
        used += len(block)
    return "\n\n".join(blocks) if blocks else "(no relevant passages found)"


def build_prompt(question, hits, max_chars=4000):
    """The two halves of what we send: standing rules, and this specific turn."""
    return {
        "system": SYSTEM_PROMPT,
        "user": USER_TEMPLATE.format(context=build_context(hits, max_chars),
                                     question=question),
    }

In [9]:
question = "What is the attention mechanism?"
prompt = build_prompt(question, retrieve_documents(question, top_k=3))

print("=" * 78)
print("SYSTEM")
print("=" * 78)
print(prompt["system"])
print()
print("=" * 78)
print("USER")
print("=" * 78)
print(prompt["user"])
print()
print("=" * 78)
print(f"total: {len(prompt['system']) + len(prompt['user'])} characters "
      f"~= {(len(prompt['system']) + len(prompt['user'])) // 4} tokens")

SYSTEM
You are a careful assistant for a technical knowledge base.

Rules:
1. Answer using ONLY the numbered context passages provided by the user.
2. If the context does not contain the answer, reply exactly:
   "I don't know based on the provided documents."
3. Cite the passage number in square brackets after each claim, like [2].
4. Never invent facts, numbers, file names or citations.
5. Be concise: two to five sentences unless the question asks for more.

USER
Context passages:
[1] (source: ai_course.pdf, page 3)
Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is projected into three vectors: a query, a key and a value. The relevance of token j to token i is the dot product between the query of i and the key of j. Those scores are divided by the square root of the head dimension to keep them numerically stable, passed through a softmax so they sum to one, and then used as weights to aver

In [10]:
hits = retrieve_documents(question, top_k=6)
print(f"{'max_chars':>10}{'passages':>10}{'context chars':>15}{'~tokens':>10}")
print("-" * 45)
for budget in (500, 1000, 2000, 4000):
    context = build_context(hits, max_chars=budget)
    print(f"{budget:>10}{context.count('(source:'):>10}{len(context):>15}"
          f"{len(context) // 4:>10}")

 max_chars  passages  context chars   ~tokens
---------------------------------------------
       500         1            536       134
      1000         1            536       134
      2000         3           1612       403
      4000         6           3226       806


## 3. Connect an LLM


In [ ]:
import ollama
MODEL = "llama3.2"
def generate_answer(prompt, model=MODEL,
                    temperature=0.0,
                    max_tokens=400):
    """Generate answer using local Ollama model."""

    messages = [
        {
            "role": "system",
            "content": prompt["system"]
        },
        {
            "role": "user",
            "content": prompt["user"]
        }
    ]

    response = ollama.chat(
        model=model,
        messages=messages,
        options={
            "temperature": temperature,
            "num_predict": max_tokens
        }
    )

    return response["message"]["content"].strip()

## 4. The complete RAG function


In [16]:
NO_CONTEXT_ANSWER = "I don't know based on the provided documents."


def format_sources(hits):
    """De-duplicated, human-readable source list."""
    seen, sources = set(), []
    for hit in hits:
        meta = hit.get("metadata", {})
        label = meta.get("source", "unknown")
        if meta.get("page"):
            label += f" page {meta['page']}"
        if label not in seen:
            seen.add(label)
            sources.append(label)
    return sources


def rag_pipeline(question, top_k=4, min_score=0.2, max_context_chars=4000,verbose=False):
    """question -> {answer, sources, hits, prompt, timings}"""
    started = time.perf_counter()

    # 1-2. retrieve
    hits = retriever.retrieve(question, top_k=top_k, min_score=min_score)
    retrieved_at = time.perf_counter()
    if verbose:
        print(f"  retrieved {len(hits)} chunks in "
              f"{(retrieved_at - started) * 1000:.0f} ms")

    # short-circuit: nothing to ground on
    if not hits:
        return {"question": question, "answer": NO_CONTEXT_ANSWER, "sources": [],
                "hits": [], "prompt": None, "grounded": False,
                "retrieval_s": retrieved_at - started, "generation_s": 0.0}

    # 3. build the prompt
    prompt = build_prompt(question, hits, max_context_chars)

    # 4. generate
    answer = generate_answer(prompt)
    finished = time.perf_counter()

    # 5. return everything, not just the answer: the hits and the prompt are
    #    what you will need the moment the answer looks wrong.
    return {"question": question, "answer": answer, "sources": format_sources(hits),
            "hits": hits, "prompt": prompt, "grounded": True,
            "retrieval_s": retrieved_at - started,
            "generation_s": finished - retrieved_at}


def print_result(result):
    print(result["answer"])
    if result["sources"]:
        print("\nSources:")
        for hit in result["hits"]:
            meta = hit["metadata"]
            page = f" page {meta['page']}" if meta.get("page") else ""
            print(f"  [{hit['rank']}] {meta['source']}{page}  (score {hit['score']:.3f})")
    else:
        print("\nSources: none - nothing in the knowledge base was relevant.")
    print(f"\n({result['retrieval_s'] * 1000:.1f} ms retrieval, "
          f"{result['generation_s']:.1f} s generation)  "
          f"<- retrieval is essentially free; the LLM is the whole latency budget")


result = rag_pipeline("What is the attention mechanism?")
print_result(result)

The attention mechanism is the operation that lets a model decide, for every token, which other tokens matter. It does this by projecting each token into three vectors: a query, a key, and a value, and then calculating the relevance of one token to another as the dot product between the query and key vectors, divided by the square root of the head dimension, passed through a softmax, and used as weights to average the value vectors. [1]

Sources:
  [1] ai_course.pdf page 3  (score 0.794)
  [2] ai_course.pdf page 2  (score 0.496)
  [3] ai_course.pdf page 3  (score 0.485)
  [4] ai_course.pdf page 3  (score 0.477)

(211.0 ms retrieval, 2.6 s generation)  <- retrieval is essentially free; the LLM is the whole latency budget


In [ ]:
result = rag_pipeline("What is the generative ai field?Answer in 1 sentence")
print_result(result)

The generative AI field refers to the development of artificial intelligence models that can generate new, original content such as text, images, or audio, based on patterns learned from large datasets.

Sources:
  [1] ai_course.pdf page 1  (score 0.385)
  [2] ai_course.pdf page 2  (score 0.316)
  [3] ai_course.pdf page 2  (score 0.293)
  [4] ai_course.pdf page 1  (score 0.288)

(177.9 ms retrieval, 1.6 s generation)  <- retrieval is essentially free; the LLM is the whole latency budget


In [20]:
for question in ["What is deep learning?",
                 "How do I get cosine similarity out of FAISS?",
                 "What are the three families of machine learning?"]:
    print("=" * 78)
    print("Q:", question)
    print("=" * 78)
    try:
        print_result(rag_pipeline(question))
    except Exception as error:
        print("generation failed:", error)
    print()

Q: What is deep learning?
Deep learning is the subset of machine learning that uses neural networks with many stacked layers. The word "deep" refers to the number of layers between the input and the output. [2]

Sources:
  [1] ai_course.pdf page 1  (score 0.636)
  [2] ai_course.pdf page 1  (score 0.591)
  [3] ai_course.pdf page 2  (score 0.564)
  [4] ai_course.pdf page 1  (score 0.559)

(159.4 ms retrieval, 2.1 s generation)  <- retrieval is essentially free; the LLM is the whole latency budget

Q: How do I get cosine similarity out of FAISS?
To get cosine similarity out of FAISS, you need to normalize both the query vector and the vectors in the index to unit length. This is done by calling `faiss.normalize_L2` on both the query vector and the vectors in the index.

[6] (source: documentation.txt)

You also need to use the inner product instead of the dot product, which can be achieved by using the cosine metric. However, FAISS does not have a built-in cosine metric, so you need to no

## 6. Evaluate the system


In [21]:
# question, expected keywords in the answer, expected source file, should refuse?
TEST_SET = [
    ("What is the attention mechanism?",
     ["quer", "key", "value"], "ai_course.pdf", False),
    ("What is deep learning?",
     ["layer"], "ai_course.pdf", False),
    ("Which index type should I use below 50,000 vectors?",
     ["flat"], "documentation.txt", False),
    ("How do I install FAISS with pip?",
     ["faiss-cpu"], "documentation.txt", False),
    ("What is the maximum value of k on GPU?",
     ["2048"], "documentation.txt", False),
    # --- questions the corpus cannot answer: the system must refuse ---
    ("How do I configure a Kubernetes ingress controller?", [], None, True),
    ("What is the boiling point of mercury?", [], None, True),
    ("Who won the 2027 Champions League final?", [], None, True),
]

def evaluate(test_set, top_k=4, min_score=0.2):
    rows = []
    for question, keywords, expected_source, should_refuse in test_set:
        hits = retriever.retrieve(question, top_k=top_k, min_score=min_score)
        retrieved_sources = {h["metadata"].get("source") for h in hits}
        hit_at_k = expected_source in retrieved_sources if expected_source else None
        try:
            result = rag_pipeline(question, top_k=top_k, min_score=min_score)
            answer = result["answer"]
        except Exception as error:
            answer = f"<generation failed: {error}>"
        refused = "don't know" in answer.lower() or "do not know" in answer.lower()
        found = [k for k in keywords if k.lower() in answer.lower()]
        if should_refuse:
            passed = refused
        else:
            passed = bool(keywords) and len(found) == len(keywords) and not refused
        rows.append({"question": question, "hit@k": hit_at_k, "refused": refused,
                     "found": found, "expected": keywords, "pass": passed,
                     "top_score": hits[0]["score"] if hits else 0.0,
                     "answer": answer})
    return rows


rows = evaluate(TEST_SET)

print(f"{'pass':>5} {'hit@k':>6} {'top':>6}  question")
print("-" * 78)
for row in rows:
    hit = "-" if row["hit@k"] is None else ("yes" if row["hit@k"] else "NO")
    print(f"{'PASS' if row['pass'] else 'FAIL':>5} {hit:>6} {row['top_score']:>6.2f}  "
          f"{row['question'][:56]}")

answerable = [r for r in rows if r["expected"]]
refusals = [r for r in rows if not r["expected"]]
print("-" * 78)
print(f"answerable questions: {sum(r['pass'] for r in answerable)}/{len(answerable)} correct, "
      f"retrieval hit@4 {sum(bool(r['hit@k']) for r in answerable)}/{len(answerable)}")
print(f"unanswerable questions: {sum(r['pass'] for r in refusals)}/{len(refusals)} correctly refused")

 pass  hit@k    top  question
------------------------------------------------------------------------------
 PASS    yes   0.79  What is the attention mechanism?
 PASS    yes   0.64  What is deep learning?
 PASS    yes   0.54  Which index type should I use below 50,000 vectors?
 PASS    yes   0.30  How do I install FAISS with pip?
 PASS    yes   0.48  What is the maximum value of k on GPU?
 PASS      -   0.00  How do I configure a Kubernetes ingress controller?
 PASS      -   0.00  What is the boiling point of mercury?
 PASS      -   0.00  Who won the 2027 Champions League final?
------------------------------------------------------------------------------
answerable questions: 5/5 correct, retrieval hit@4 5/5
unanswerable questions: 3/3 correctly refused


In [22]:
# Read the failures. This is where the learning is - a table of PASS/FAIL tells
# you nothing about WHY.
for row in rows:
    if not row["pass"]:
        print("=" * 78)
        print("Q:", row["question"])
        print(f"   retrieval hit@k: {row['hit@k']}   top score: {row['top_score']:.3f}")
        print(f"   expected keywords: {row['expected']}   found: {row['found']}")
        print(f"   answer: {row['answer'][:400]}")
        print()

print("Diagnose each one with the table above:")
print("  hit@k = yes but wrong answer   -> generation problem (prompt / model)")
print("  hit@k = NO                    -> retrieval problem (chunking / top_k / embedding)")
print("  refused but corpus has it     -> min_score too high, or phrasing mismatch")
print("\nIf you are on hf_local, expect several generation failures: flan-t5-base is")
print("a 250M-parameter model and will not follow a five-rule system prompt reliably.")
print("That is itself a finding - model capability is part of your RAG design.")

Diagnose each one with the table above:
  hit@k = yes but wrong answer   -> generation problem (prompt / model)
  hit@k = NO                    -> retrieval problem (chunking / top_k / embedding)
  refused but corpus has it     -> min_score too high, or phrasing mismatch

If you are on hf_local, expect several generation failures: flan-t5-base is
a 250M-parameter model and will not follow a five-rule system prompt reliably.
That is itself a finding - model capability is part of your RAG design.


### Experiment 6.1 — the control experiment: same model, no retrieval

In [23]:
question = ("According to our benchmarks, what recall does IndexIVFFlat reach "
            "at nprobe=8, and how many milliseconds per query does it take?")

print("=" * 78)
print("WITHOUT RAG (model knowledge only)")
print("=" * 78)
try:
    print(generate_answer({"system": "You are a helpful assistant.",
                           "user": question}, max_tokens=200))
except Exception as error:
    print("generation failed:", error)

print("\n" + "=" * 78)
print("WITH RAG (grounded in documentation.txt)")
print("=" * 78)
try:
    print_result(rag_pipeline(question))
except Exception as error:
    print("generation failed:", error)

print("\nThe correct answer is in section 10 of documentation.txt: 94% recall at")
print("0.9 ms per query. Compare that to what the ungrounded model said. Whatever")
print("it produced - a refusal, a plausible number, a confident table - it had no")
print("way of knowing, because we measured those numbers ourselves.")

WITHOUT RAG (model knowledge only)
I don't have any information about the specific benchmark results for IndexIVFFlat or its performance metrics. Can you provide more context or details about the benchmarking process used to obtain these results? I'll do my best to help you find the answer.

WITH RAG (grounded in documentation.txt)
According to the provided context passages, IndexIVFFlat reaches 94% recall at nprobe=8, and it takes 0.9 milliseconds per query. [1] [2]

Sources:
  [1] documentation.txt  (score 0.763)
  [2] documentation.txt  (score 0.681)
  [3] documentation.txt  (score 0.500)
  [4] documentation.txt  (score 0.477)

(113.8 ms retrieval, 1.7 s generation)  <- retrieval is essentially free; the LLM is the whole latency budget

The correct answer is in section 10 of documentation.txt: 94% recall at
0.9 ms per query. Compare that to what the ungrounded model said. Whatever
it produced - a refusal, a plausible number, a confident table - it had no
way of knowing, because we m

In [24]:
# Retrieval-only sweep: fast, no LLM calls, and it isolates the retrieval half.
print(f"{'top_k':>6}{'min_score':>11}{'hit@k':>8}{'false ctx':>11}")
print("-" * 36)
for top_k in (1, 2, 4, 8):
    for min_score in (0.0, 0.2, 0.35):
        hit_count = false_context = 0
        for question, _, expected_source, should_refuse in TEST_SET:
            hits = retriever.retrieve(question, top_k=top_k, min_score=min_score)
            sources = {h["metadata"].get("source") for h in hits}
            if expected_source and expected_source in sources:
                hit_count += 1
            if should_refuse and hits:      # context supplied for an unanswerable question
                false_context += 1
        answerable = sum(1 for t in TEST_SET if t[2])
        unanswerable = sum(1 for t in TEST_SET if t[3])
        print(f"{top_k:>6}{min_score:>11}{hit_count:>4}/{answerable:<3}"
              f"{false_context:>7}/{unanswerable}")

 top_k  min_score   hit@k  false ctx
------------------------------------
     1        0.0   5/5        3/3
     1        0.2   5/5        0/3
     1       0.35   4/5        0/3
     2        0.0   5/5        3/3
     2        0.2   5/5        0/3
     2       0.35   4/5        0/3
     4        0.0   5/5        3/3
     4        0.2   5/5        0/3
     4       0.35   4/5        0/3
     8        0.0   5/5        3/3
     8        0.2   5/5        0/3
     8       0.35   4/5        0/3


---
## 7. Packaging it: `rag_pipeline.py`

The whole session now lives in a module next to `rag_utils.py` and `rag_store.py`:

- `Retriever` - question -> chunks, with `top_k`, `min_score` and a source filter
- `build_context` / `build_prompt` - numbered, budgeted, source-tagged context
- `generate_answer` / `OllamaGenerator` - the LLM behind a single call
- `RAGPipeline.ask` - retrieve -> prompt -> generate -> cite, returning the hits
  and the prompt alongside the answer
- `evaluate` / `sweep_retrieval` - the scoreboard, and the knobs, measured

Three files, three sessions: `rag_utils` makes chunks, `rag_store` makes them
searchable, `rag_pipeline` answers questions with them.

In [ ]:
# Session 3, packaged: retriever + prompt + generator + evaluation.
# Everything below runs against rag_pipeline.py, not the definitions above.
import importlib

import rag_pipeline
importlib.reload(rag_pipeline)                   # pick up edits without restarting

from rag_pipeline import (TEST_SET, build_pipeline, evaluate, format_evaluation,
                          format_failures, print_result, sweep_retrieval)

rag = build_pipeline()                           # reuses Session 2's saved store
print(f"knowledge base: {len(rag.retriever.store)} chunks, "
      f"{rag.retriever.store.dimension} dims, model {rag.retriever.store.model_name}")
print()

print_result(rag.ask("What is the attention mechanism?"))
print()

# Retrieval-only sweep: no LLM calls, so it is fast enough to actually use.
print(sweep_retrieval(rag.retriever))
print()

rows = evaluate(rag, TEST_SET)
print(format_evaluation(rows))
print()
print(format_failures(rows))